In [1]:
def calcular_long(x,E):
    cam = np.array(x)
    dist = 0
    for i in range(len(cam)-1):
        dist += E[cam[i],cam[i+1]]
    
    return dist

In [2]:
import numpy as np
import pandas as pd
import time

def sistemaHormigas(rho=0.1, maxIter=500, Q=1, tipo_deposito=1):
    E = pd.read_csv("gr17.csv",header=None).to_numpy()

    N = 10
    sigma = np.random.random(E.shape)
    it = 0

    p = {}
    mejor_dist = np.inf
    mejor_cam = []

    while True:
        dist_cam = {}
        for k in range(N):
            p[k] = [np.random.randint(0,E.shape[1])]
            ciudades_x_visitar = np.ones(E.shape[0])
            ciudades_x_visitar[p[k][0]] = 0
            
            while (ciudades_x_visitar == 1).any():
                prob = np.zeros(E.shape[0])
                # for j in range(len(ciudades_x_visitar==1)):
                #     i = p[k][-1]
                #     sum = 0
                #     for u in range(len(ciudades_x_visitar==1)):
                #         if ciudades_x_visitar[u]==1:
                #             sum += sigma[i,u]* ( 1/E[i,u])
                #     if ciudades_x_visitar[j]==1:
                #         prob[j] = sigma[i,j]* ( 1/E[i,j]) / sum

                i = p[k][-1]
                suma = 0
                for u in range(len(ciudades_x_visitar)):
                    if ciudades_x_visitar[u]==1:
                        suma += sigma[i,u]* ( 1/E[i,u])
                
                for j in range(len(ciudades_x_visitar)):
                    if ciudades_x_visitar[j]==1:
                        prob[j] = sigma[i,j]* ( 1/E[i,j]) / suma

                
                
                prob_elegir = np.random.random()
                ind_elegir = -1
                suma_prob = 0
                for j in range(len(prob)):
                    if(prob[j] != 0):
                        suma_prob += prob[j]
                        ind_elegir = j
                        if (suma_prob >= prob_elegir):
                            break
                    # if(prob[j] >= prob_elegir):
                    #     ind_elegir = j
                    #     break
                p[k].append(ind_elegir)
                ciudades_x_visitar[ind_elegir] = 0

            p[k].append(p[k][0])
            dist_cam[k] = calcular_long(p[k],E)
            if dist_cam[k] < mejor_dist:
                mejor_dist = dist_cam[k]
                mejor_cam = p[k].copy()

        k_1 = p[0]
        iguales = True
        for k in range(1,N):
            if p[k] != k_1:
                iguales = False
                break
        if iguales or it >= maxIter: break 


        # k_1 = p[0]
        # iguales = False
        # for k in range(1,N):
        #     if (p[k]==k_1).all():
        #         iguales = True
        #     else:
        #         iguales = False

        # if iguales: break
        # para cada conexion (i,j):
        # reducir por evaporacion las feromonas
        for i in range(sigma.shape[0]):
            for j in range(sigma.shape[1]):
                sigma[i,j] = (1-rho)*sigma[i,j]

        # depositar feromonas
        for k in range(N):
            if tipo_deposito == 1:
                delta_sigma_k = Q/dist_cam[k]
            elif tipo_deposito == 2:
                delta_sigma_k = Q
            

            for idx in range(len(p[k])-1):
                i, j = p[k][idx], p[k][idx+1]
                if tipo_deposito == 3:
                    delta_sigma_k = Q/E[i,j]
                    
                sigma[i,j] += delta_sigma_k
                sigma[j,i] += delta_sigma_k

        it += 1
        # for i in range(sigma.shape[0]):
        #     for j in range(sigma.shape[1]):  
        #         delta_sigma_k = 0
        #         for k in range(N): 
        #             idx_i = p[k].index(i)
        #             idx_j = p[k].index(j)

        #             if (idx_i+1 < sigma.shape[0] and idx_i+1 == idx_j): # (i,j) pertenece al camino p[k]
        #                 delta_sigma_k += Q/calcular_long(p[k], E)

    return (it, mejor_cam, mejor_dist)


            

In [3]:

valores_rho = [0.1, 0.3, 0.5, 0.8] 
valores_tipo = [1, 2, 3]
max_iteraciones = 500

nombres_deposito = {1: 'Global', 2: 'Uniforme', 3: 'Local'}
resultados = []

for rho in valores_rho:
    for tipo in valores_tipo:
        print(f"Evaporación={rho}, Tipo depósito={nombres_deposito[tipo]}")

        t_0 = time.perf_counter()
        it, mejor_cam, mejor_dist = sistemaHormigas(rho=rho, maxIter=max_iteraciones, Q=1, tipo_deposito=tipo)
        t_fin = time.perf_counter()
        t_ejecucion = t_fin - t_0
        
        # Guardar los resultados como un diccionario
        resultados.append({
            'Rho (Evaporación)': rho,
            'Tipo de Depósito': nombres_deposito[tipo],
            'Mejor distancia': mejor_dist,
            'Tiempo de ejecución [s]': round(t_ejecucion,3),
            'Iteración final': it,
            'Camino': str(mejor_cam)
        })
    print("\n")


tabla = pd.DataFrame(resultados)    # formatear los resultados como una tabla
tabla_ordenada = tabla.sort_values(by='Mejor distancia', ascending=True)    # ordenar por valor de costo en distancia

display(tabla_ordenada)


Evaporación=0.1, Tipo depósito=Global
Evaporación=0.1, Tipo depósito=Uniforme
Evaporación=0.1, Tipo depósito=Local


Evaporación=0.3, Tipo depósito=Global
Evaporación=0.3, Tipo depósito=Uniforme
Evaporación=0.3, Tipo depósito=Local


Evaporación=0.5, Tipo depósito=Global
Evaporación=0.5, Tipo depósito=Uniforme
Evaporación=0.5, Tipo depósito=Local


Evaporación=0.8, Tipo depósito=Global
Evaporación=0.8, Tipo depósito=Uniforme
Evaporación=0.8, Tipo depósito=Local




,Rho (Evaporación),Tipo de Depósito,Mejor distancia,Tiempo de ejecución [s],Iteración final,Camino
0,0.1,Global,2085,1.232,500,"[1, 4, 8, 11, 15, 0, 3, 12, 6, 7, 5, 16, 13, 1..."
1,0.1,Uniforme,2085,1.209,500,"[8, 11, 15, 0, 3, 12, 6, 7, 5, 16, 13, 14, 2, ..."
2,0.1,Local,2085,1.302,500,"[14, 13, 16, 5, 7, 6, 12, 3, 0, 15, 11, 8, 4, ..."
3,0.3,Global,2085,1.172,500,"[16, 5, 7, 6, 12, 3, 0, 15, 11, 8, 4, 1, 9, 10..."
4,0.3,Uniforme,2085,1.129,500,"[2, 14, 13, 16, 5, 7, 6, 12, 3, 0, 15, 11, 8, ..."
6,0.5,Global,2085,1.203,500,"[11, 15, 0, 3, 12, 6, 7, 5, 16, 13, 14, 2, 10,..."
7,0.5,Uniforme,2085,1.111,500,"[14, 13, 16, 5, 7, 6, 12, 3, 0, 15, 11, 8, 4, ..."
8,0.5,Local,2085,1.163,500,"[10, 2, 14, 13, 16, 5, 7, 6, 12, 3, 0, 15, 11,..."
11,0.8,Local,2085,1.257,500,"[1, 9, 10, 2, 14, 13, 16, 5, 7, 6, 12, 3, 0, 1..."
9,0.8,Global,2085,1.170,500,"[4, 8, 11, 15, 0, 3, 12, 6, 7, 5, 16, 13, 14, ..."
